# PPO D69 消融 — 无时机锚(`ppo_8192_s0`)

配置: 8192批量 / lr 1e-4 / 升档=完成率1.0且延误≤fcfs(k=1.0) / best-rescue / c12 口径。**与 anchor 版唯一差异 = 没有 `--bc-anchor-beta`**。两个 notebook 在服务器各开一个同时跑。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## 训练(重跑本格 = 断点续训)

In [ ]:
!{PREFIX} scripts/13_train_ppo.py \
    --run-tag ppo_8192_s0 \
    --encoder gnn --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --control-classes 12 \
    --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999 \
    --promote-at 1.0 --promote-delay-k 1.0 \
    --train-eval-days 6 --iters 120 --steps-per-iter 8192 --lr 1e-4 --eval-every 5 \
    --early-stop-patience 8 --best-rescue --resume


## 对比(两份都有结果后, 在任一 notebook 跑本格即可)

In [ ]:
import json, torch
from pathlib import Path
rep=Path('artifacts/reports')
def load(tag):
    if (rep/f'{tag}.json').exists(): return json.loads((rep/f'{tag}.json').read_text())
    if (rep/f'{tag}_last.pt').exists(): return torch.load(rep/f'{tag}_last.pt',map_location='cpu',weights_only=False)
    return None
runs=[('8192 无锚','ppo_8192_s0'),('8192+时机锚','ppo_8192_anchor_s0'),('旧2048','ppo_p12_s0')]
loaded={n:load(t) for n,t in runs}
base=next((d.get('baselines') for d in loaded.values() if d and d.get('baselines')),{})
print(f"{'run':<12}{'best完成':>9}{'best延误':>11}{'best_rew':>10}{'@it':>5}")
for k in ['fcfs','lookup_most_frequent','greedy_set','signaller_replay']:
    if k in base:
        b=base[k]; print(f"{k:<12}{b['eval_completion_rate']:>9.3f}{b['eval_weighted_secondary_delay_s']:>11.0f}{b['eval_episode_reward']:>10.0f}{'':>5}")
for name,_ in runs:
    d=loaded.get(name)
    if not d or not d.get('best'): print(f'{name:<12}(missing)'); continue
    b=d['best']; print(f"{name:<12}{b.get('completion',0):>9.3f}{b.get('delay',0):>11.0f}{b.get('reward',0):>10.0f}{b.get('iter',-1):>5}")
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(11,3.2))
    for name,_ in runs:
        d=loaded.get(name)
        if d and d.get('evals'):
            ev=d['evals']
            axes[0].plot([e['iter'] for e in ev],[e['eval_completion_rate'] for e in ev],marker='.',label=name)
            axes[1].plot([e['iter'] for e in ev],[e['eval_weighted_secondary_delay_s'] for e in ev],marker='.',label=name)
    axes[0].axhline(1.0,ls='--',c='g',lw=.8); axes[0].set_title('val 完成率'); axes[0].legend(fontsize=8)
    if 'fcfs' in base: axes[1].axhline(base['fcfs']['eval_weighted_secondary_delay_s'],ls='--',c='b',lw=.8,label='fcfs')
    if 'greedy_set' in base: axes[1].axhline(base['greedy_set']['eval_weighted_secondary_delay_s'],ls='--',c='r',lw=.8,label='always-set')
    axes[1].set_yscale('log'); axes[1].set_title('val 延误(log)'); axes[1].legend(fontsize=7)
    plt.tight_layout(); plt.savefig('artifacts/reports/d69_compare.png',dpi=110)
    print('fig -> artifacts/reports/d69_compare.png')
except Exception as e: print('plot skip',e)
